In [7]:
import pandas as pd
import numpy as np


In [8]:
df = pd.read_excel("../data/raw/UCI_Credit_Card.xlsx")
df.columns = df.columns.str.replace('# ', '', regex=False).str.strip()
df = df.drop(columns=["ID"])

In [4]:
#rebuild X, y train/test split
y = df["default.payment.next.month"]
X = df.drop(columns=["default.payment.next.month"])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000))
])

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('logreg', LogisticRegression(max_iter=2000))])

In [6]:
#create batches
batch_size = 5000
batches = []

for i in range(0, len(X_test), batch_size):
    batches.append(X_test.iloc[i:i+batch_size])

baseline_batch = batches[0].copy()
drifted_batch = batches[1].copy()

In [7]:
#inject drift
drifted_batch["LIMIT_BAL"] = drifted_batch["LIMIT_BAL"] * 1.4
drifted_batch["PAY_0"] = drifted_batch["PAY_0"] + 1

In [8]:
#PSI function + compute PSI report + drift levels
def psi(expected, actual, buckets=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)
    if len(breakpoints) <= 2:
        return 0.0

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    expected_perc = expected_counts / len(expected)
    actual_perc = actual_counts / len(actual)

    expected_perc = np.where(expected_perc == 0, 1e-6, expected_perc)
    actual_perc = np.where(actual_perc == 0, 1e-6, actual_perc)

    return np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))

psi_report = {}
for col in baseline_batch.columns:
    psi_report[col] = psi(baseline_batch[col], drifted_batch[col])

psi_df = pd.DataFrame.from_dict(psi_report, orient="index", columns=["PSI"])
psi_df = psi_df.sort_values("PSI", ascending=False)

def categorize_psi(v):
    if v < 0.10:
        return "Stable"
    elif v < 0.25:
        return "Moderate Drift"
    else:
        return "Severe Drift"

psi_df["Drift_Level"] = psi_df["PSI"].apply(categorize_psi)

severe_count = (psi_df["Drift_Level"] == "Severe Drift").sum()
moderate_count = (psi_df["Drift_Level"] == "Moderate Drift").sum()

severe_count, moderate_count
 

(2, 0)

In [9]:
#prediction drift PSI
baseline_proba = pipeline.predict_proba(baseline_batch)[:, 1]
drifted_proba = pipeline.predict_proba(drifted_batch)[:, 1]

psi_pred = psi(baseline_proba, drifted_proba)
psi_pred

0.5454011381130935

In [10]:
#performance drift (AUC by batch)
from sklearn.metrics import roc_auc_score

y_test_batches = []
for i in range(0, len(y_test), batch_size):
    y_test_batches.append(y_test.iloc[i:i+batch_size])

baseline_y = y_test_batches[0]
drifted_y = y_test_batches[1]

auc_baseline = roc_auc_score(baseline_y, baseline_proba[:len(baseline_y)])
auc_drifted = roc_auc_score(drifted_y, drifted_proba[:len(drifted_y)])

auc_baseline, auc_drifted

(0.7100416888711254, 0.6944393423196562)

TASK 14- Agent decision logic

In [11]:
#build agent decision output
def agent_decision(feature_severe, pred_psi, auc_baseline, auc_drifted):
    
    performance_drop = auc_baseline - auc_drifted
    
    if feature_severe >= 2 and pred_psi > 0.25:
        decision = "Escalate: Significant feature and prediction drift detected."
    
    elif performance_drop > 0.05:
        decision = "Investigate: Performance degradation detected."
    
    elif pred_psi > 0.25:
        decision = "Monitor Closely: Prediction distribution shifted."
    
    else:
        decision = "Stable: No immediate action required."
    
    return decision


decision_output = agent_decision(
    severe_count,
    psi_pred,
    auc_baseline,
    auc_drifted
)

print("Agent Decision:", decision_output)

Agent Decision: Escalate: Significant feature and prediction drift detected.


TASK 15- Generate a Structured Incident Report

In [12]:
#create a "top drift features" list
top_drift_features = (
    psi_df.reset_index()
          .rename(columns={"index": "feature"})
          .sort_values("PSI", ascending=False)
          .head(5)[["feature", "PSI", "Drift_Level"]]
)

top_drift_features

,feature,PSI,Drift_Level
0,PAY_0,1.983257,Severe Drift
1,LIMIT_BAL,0.434594,Severe Drift
2,BILL_AMT6,0.014470,Stable
3,PAY_AMT5,0.012712,Stable
4,BILL_AMT5,0.010068,Stable


In [14]:
#recompute batch risk
def compute_batch_risk(severe_count, moderate_count):
    if severe_count >= 2:
        return "High Risk"
    elif severe_count == 1 or moderate_count >= 2:
        return "Medium Risk"
    elif moderate_count == 1:
        return "Low Risk"
    else:
        return "Stable"

batch_risk = compute_batch_risk(severe_count, moderate_count)

batch_risk

'High Risk'

In [16]:
decision_output = agent_decision(
    severe_count,
    psi_pred,
    auc_baseline,
    auc_drifted
)

decision_output

'Escalate: Significant feature and prediction drift detected.'

In [18]:
#build incident report dictionary
performance_drop = auc_baseline - auc_drifted

incident = {
    "incident_type": "Model Monitoring Alert",
    "batch_risk_level": batch_risk,  # from Task 11
    "decision": decision_output,     # from Task 14
    "summary_metrics": {
        "severe_feature_drift_count": int(severe_count),
        "moderate_feature_drift_count": int(moderate_count),
        "prediction_psi": float(psi_pred),
        "auc_baseline": float(auc_baseline),
        "auc_drifted": float(auc_drifted),
        "auc_drop": float(performance_drop),
    },
    "top_drift_features": top_drift_features.to_dict(orient="records"),
}

incident

{'incident_type': 'Model Monitoring Alert',
 'batch_risk_level': 'High Risk',
 'decision': 'Escalate: Significant feature and prediction drift detected.',
 'summary_metrics': {'severe_feature_drift_count': 2,
  'moderate_feature_drift_count': 0,
  'prediction_psi': 0.5454011381130935,
  'auc_baseline': 0.7100416888711254,
  'auc_drifted': 0.6944393423196562,
  'auc_drop': 0.015602346551469193},
 'top_drift_features': [{'feature': 'PAY_0',
   'PSI': 1.983256732331724,
   'Drift_Level': 'Severe Drift'},
  {'feature': 'LIMIT_BAL',
   'PSI': 0.434594027897233,
   'Drift_Level': 'Severe Drift'},
  {'feature': 'BILL_AMT6',
   'PSI': 0.014470424699483356,
   'Drift_Level': 'Stable'},
  {'feature': 'PAY_AMT5',
   'PSI': 0.012712095696392242,
   'Drift_Level': 'Stable'},
  {'feature': 'BILL_AMT5',
   'PSI': 0.010067571277132776,
   'Drift_Level': 'Stable'}]}

In [19]:
#printing it
import json
print(json.dumps(incident, indent=2))

{
  "incident_type": "Model Monitoring Alert",
  "batch_risk_level": "High Risk",
  "decision": "Escalate: Significant feature and prediction drift detected.",
  "summary_metrics": {
    "severe_feature_drift_count": 2,
    "moderate_feature_drift_count": 0,
    "prediction_psi": 0.5454011381130935,
    "auc_baseline": 0.7100416888711254,
    "auc_drifted": 0.6944393423196562,
    "auc_drop": 0.015602346551469193
  },
  "top_drift_features": [
    {
      "feature": "PAY_0",
      "PSI": 1.983256732331724,
      "Drift_Level": "Severe Drift"
    },
    {
      "feature": "LIMIT_BAL",
      "PSI": 0.434594027897233,
      "Drift_Level": "Severe Drift"
    },
    {
      "feature": "BILL_AMT6",
      "PSI": 0.014470424699483356,
      "Drift_Level": "Stable"
    },
    {
      "feature": "PAY_AMT5",
      "PSI": 0.012712095696392242,
      "Drift_Level": "Stable"
    },
    {
      "feature": "BILL_AMT5",
      "PSI": 0.010067571277132776,
      "Drift_Level": "Stable"
    }
  ]
}


TASK 16- Human-in-the-loop approval gate

In [20]:
def recommend_actions(decision, batch_risk_level):
    # Default: no action
    actions = []

    if "Escalate" in decision or batch_risk_level in ["High Risk"]:
        actions = [
            "Notify model owner + risk team",
            "Run deeper diagnostics (SHAP / feature importance shift)",
            "Freeze automated decisioning if high-stakes",
            "Collect additional validation data",
            "Prepare retraining plan (do not retrain automatically)"
        ]
    elif "Investigate" in decision:
        actions = [
            "Check recent data pipeline changes",
            "Validate labels and ground truth lag",
            "Run performance evaluation on latest batch",
            "Consider threshold adjustment temporarily"
        ]
    elif "Monitor Closely" in decision:
        actions = [
            "Increase monitoring frequency",
            "Track prediction distribution daily",
            "Set alert if PSI crosses 0.25 again"
        ]
    else:
        actions = ["No action required - continue monitoring"]

    return actions


incident["recommended_actions"] = recommend_actions(
    incident["decision"],
    incident["batch_risk_level"]
)

incident["recommended_actions"]

['Notify model owner + risk team',
 'Run deeper diagnostics (SHAP / feature importance shift)',
 'Freeze automated decisioning if high-stakes',
 'Collect additional validation data',
 'Prepare retraining plan (do not retrain automatically)']

In [21]:
#add approval requirement (HITL)
def requires_approval(batch_risk_level, decision):
    # Human approval gate for any serious situation
    if batch_risk_level in ["High Risk", "Medium Risk"]:
        return True
    if "Escalate" in decision or "Investigate" in decision:
        return True
    return False


incident["requires_human_approval"] = requires_approval(
    incident["batch_risk_level"],
    incident["decision"]
)

incident["requires_human_approval"]

True

In [22]:
#simulate approval decision
# Simulated approval (in real systems this would be a ticket / workflow tool)
incident["human_approval_status"] = "PENDING" if incident["requires_human_approval"] else "NOT_REQUIRED"

incident["human_approval_status"]

'PENDING'

TASK 17- Audit Logging (save incident to file)

In [23]:
import os
from datetime import datetime

os.makedirs("../reports/incidents", exist_ok=True)

In [24]:
#save incident JSON with timestamp filename
import json

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_path = f"../reports/incidents/incident_{timestamp}.json"

with open(file_path, "w") as f:
    json.dump(incident, f, indent=2)

file_path

'../reports/incidents/incident_20260213_203852.json'

In [25]:
#read it back to confirm
with open(file_path, "r") as f:
    loaded_incident = json.load(f)

loaded_incident["incident_type"], loaded_incident["human_approval_status"]

('Model Monitoring Alert', 'PENDING')

In [19]:
# wrapper function
def run_monitoring():

    # --- Load & Clean Data ---
    df = pd.read_excel("../data/raw/UCI_Credit_Card.xlsx")
    df.columns = df.columns.str.replace('# ', '', regex=False).str.strip()
    df = df.drop(columns=["ID"])

    # --- Train/Test Split ---
    y = df["default.payment.next.month"]
    X = df.drop(columns=["default.payment.next.month"])

    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # --- Model ---
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=2000))
    ])
    pipeline.fit(X_train, y_train)

    # --- Create Batches ---
    batch_size = 5000
    batches = []
    for i in range(0, len(X_test), batch_size):
        batches.append(X_test.iloc[i:i+batch_size])

    baseline_batch = batches[0].copy()
    drifted_batch = batches[1].copy()

    # --- Inject Drift ---
    drifted_batch["LIMIT_BAL"] *= 1.4
    drifted_batch["PAY_0"] += 1

    # --- PSI Function ---
    def psi(expected, actual, buckets=10):
        expected = pd.Series(expected).dropna()
        actual = pd.Series(actual).dropna()

        breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
        breakpoints = np.unique(breakpoints)
        if len(breakpoints) <= 2:
            return 0.0

        expected_counts = np.histogram(expected, bins=breakpoints)[0]
        actual_counts = np.histogram(actual, bins=breakpoints)[0]

        expected_perc = expected_counts / len(expected)
        actual_perc = actual_counts / len(actual)

        expected_perc = np.where(expected_perc == 0, 1e-6, expected_perc)
        actual_perc = np.where(actual_perc == 0, 1e-6, actual_perc)

        return np.sum((actual_perc - expected_perc) * np.log(actual_perc / expected_perc))

    # --- Feature Drift ---
    psi_report = {col: psi(baseline_batch[col], drifted_batch[col]) for col in baseline_batch.columns}
    psi_df = pd.DataFrame.from_dict(psi_report, orient="index", columns=["PSI"])

    def categorize(v):
        if v < 0.10:
            return "Stable"
        elif v < 0.25:
            return "Moderate Drift"
        else:
            return "Severe Drift"

    psi_df["Drift_Level"] = psi_df["PSI"].apply(categorize)

    severe = int((psi_df["Drift_Level"] == "Severe Drift").sum())
    moderate = int((psi_df["Drift_Level"] == "Moderate Drift").sum())

    # --- Top drift features (Task 19) ---
    top_drift_features = (
        psi_df.reset_index()
              .rename(columns={"index": "feature"})
              .sort_values("PSI", ascending=False)
              .head(5)[["feature", "PSI", "Drift_Level"]]
              .to_dict(orient="records")
    )

    # --- Prediction Drift ---
    baseline_proba = pipeline.predict_proba(baseline_batch)[:, 1]
    drifted_proba = pipeline.predict_proba(drifted_batch)[:, 1]
    prediction_psi = float(psi(baseline_proba, drifted_proba))

    # --- Performance Drift ---
    from sklearn.metrics import roc_auc_score

    y_batches = []
    for i in range(0, len(y_test), batch_size):
        y_batches.append(y_test.iloc[i:i+batch_size])

    auc_base = float(roc_auc_score(y_batches[0], baseline_proba[:len(y_batches[0])]))
    auc_new = float(roc_auc_score(y_batches[1], drifted_proba[:len(y_batches[1])]))
    perf_drop = float(auc_base - auc_new)

    # --- Risk Logic ---
    def compute_batch_risk(severe_count, moderate_count):
        if severe_count >= 2:
            return "High Risk"
        elif severe_count == 1 or moderate_count >= 2:
            return "Medium Risk"
        elif moderate_count == 1:
            return "Low Risk"
        else:
            return "Stable"

    batch_level = compute_batch_risk(severe, moderate)

    def agent_decision(feature_severe, pred_psi, auc_baseline, auc_drifted):
        performance_drop = auc_baseline - auc_drifted
        if feature_severe >= 2 and pred_psi > 0.25:
            return "Escalate: Significant feature and prediction drift detected."
        elif performance_drop > 0.05:
            return "Investigate: Performance degradation detected."
        elif pred_psi > 0.25:
            return "Monitor Closely: Prediction distribution shifted."
        else:
            return "Stable: No immediate action required."

    decision = agent_decision(severe, prediction_psi, auc_base, auc_new)

    # --- Actions + HITL (Task 19) ---
    def recommend_actions(decision, batch_risk_level):
        if "Escalate" in decision or batch_risk_level == "High Risk":
            return [
                "Notify model owner + risk team",
                "Run deeper diagnostics (SHAP / feature importance shift)",
                "Freeze automated decisioning if high-stakes",
                "Collect additional validation data",
                "Prepare retraining plan (do not retrain automatically)"
            ]
        elif "Investigate" in decision:
            return [
                "Check recent data pipeline changes",
                "Validate labels and ground truth lag",
                "Run performance evaluation on latest batch",
                "Consider threshold adjustment temporarily"
            ]
        elif "Monitor Closely" in decision:
            return [
                "Increase monitoring frequency",
                "Track prediction distribution daily",
                "Set alert if PSI crosses 0.25 again"
            ]
        else:
            return ["No action required - continue monitoring"]

    def requires_approval(batch_risk_level, decision):
        if batch_risk_level in ["High Risk", "Medium Risk"]:
            return True
        if "Escalate" in decision or "Investigate" in decision:
            return True
        return False

    actions = recommend_actions(decision, batch_level)
    approval_required = requires_approval(batch_level, decision)
    approval_status = "PENDING" if approval_required else "NOT_REQUIRED"

    # --- Final Incident ---
    incident_report = {
        "incident_type": "Model Monitoring Alert",
        "batch_risk_level": batch_level,
        "decision": decision,
        "summary_metrics": {
            "severe_feature_drift_count": severe,
            "moderate_feature_drift_count": moderate,
            "prediction_psi": prediction_psi,
            "auc_baseline": auc_base,
            "auc_drifted": auc_new,
            "auc_drop": perf_drop,
        },
        "top_drift_features": top_drift_features,
        "recommended_actions": actions,
        "requires_human_approval": approval_required,
        "human_approval_status": approval_status
    }
      # --- Auto Audit Logging (Task 20) ---
    os.makedirs("../reports/incidents", exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_path = f"../reports/incidents/incident_{timestamp}.json"

    with open(file_path, "w") as f:
        json.dump(incident_report, f, indent=2)

    incident_report["log_file_path"] = file_path

    return incident_report


# run it (outside the function)
incident_final = run_monitoring()

import json
print(json.dumps(incident_final, indent=2))


{
  "incident_type": "Model Monitoring Alert",
  "batch_risk_level": "High Risk",
  "decision": "Escalate: Significant feature and prediction drift detected.",
  "summary_metrics": {
    "severe_feature_drift_count": 2,
    "moderate_feature_drift_count": 0,
    "prediction_psi": 0.5454011381130935,
    "auc_baseline": 0.7100416888711254,
    "auc_drifted": 0.6944393423196562,
    "auc_drop": 0.015602346551469193
  },
  "top_drift_features": [
    {
      "feature": "PAY_0",
      "PSI": 1.983256732331724,
      "Drift_Level": "Severe Drift"
    },
    {
      "feature": "LIMIT_BAL",
      "PSI": 0.434594027897233,
      "Drift_Level": "Severe Drift"
    },
    {
      "feature": "BILL_AMT6",
      "PSI": 0.014470424699483356,
      "Drift_Level": "Stable"
    },
    {
      "feature": "PAY_AMT5",
      "PSI": 0.012712095696392242,
      "Drift_Level": "Stable"
    },
    {
      "feature": "BILL_AMT5",
      "PSI": 0.010067571277132776,
      "Drift_Level": "Stable"
    }
  ],
  "reco

In [20]:
incident_final = run_monitoring()

import json
print(json.dumps(incident_final, indent=2))

{
  "incident_type": "Model Monitoring Alert",
  "batch_risk_level": "High Risk",
  "decision": "Escalate: Significant feature and prediction drift detected.",
  "summary_metrics": {
    "severe_feature_drift_count": 2,
    "moderate_feature_drift_count": 0,
    "prediction_psi": 0.5454011381130935,
    "auc_baseline": 0.7100416888711254,
    "auc_drifted": 0.6944393423196562,
    "auc_drop": 0.015602346551469193
  },
  "top_drift_features": [
    {
      "feature": "PAY_0",
      "PSI": 1.983256732331724,
      "Drift_Level": "Severe Drift"
    },
    {
      "feature": "LIMIT_BAL",
      "PSI": 0.434594027897233,
      "Drift_Level": "Severe Drift"
    },
    {
      "feature": "BILL_AMT6",
      "PSI": 0.014470424699483356,
      "Drift_Level": "Stable"
    },
    {
      "feature": "PAY_AMT5",
      "PSI": 0.012712095696392242,
      "Drift_Level": "Stable"
    },
    {
      "feature": "BILL_AMT5",
      "PSI": 0.010067571277132776,
      "Drift_Level": "Stable"
    }
  ],
  "reco

TASK 20- Auto-Save the incident log inside

In [21]:
import os
from datetime import datetime
import json

In [23]:
incident_final = run_monitoring()
incident_final["log_file_path"]

'../reports/incidents/incident_20260215_160947.json'

In [24]:
with open(incident_final["log_file_path"], "r") as f:
    saved = json.load(f)

saved.keys()


dict_keys(['incident_type', 'batch_risk_level', 'decision', 'summary_metrics', 'top_drift_features', 'recommended_actions', 'requires_human_approval', 'human_approval_status'])